# Socratic-OT | Phase 1: Knowledge Base → ChromaDB

**Project:** Socratic-OT Multimodal AI Tutor  
**Team:** Vidhyadhari Bandaru, Richie Ilavarapu  

This notebook builds the retrieval backbone of the Socratic-OT tutor.

### What this notebook does
1. Loads all **997 text chunks** from OpenStax Anatomy & Physiology 2e (28 chapters — full textbook)
2. Generates **dense embeddings** using `all-MiniLM-L6-v2` (local, free, no API key needed)
3. Ingests everything into a **ChromaDB** vector store with full metadata
4. Validates the store and runs test retrieval queries
5. Persists the database to Google Drive for reuse in subsequent notebooks

### Architecture context
```
[text_chunks_full 2.csv]  →  [all-MiniLM-L6-v2 embeddings]  →  [ChromaDB]
                                                                      ↓
                                                         Retrieval layer (Phase 2)
```

---
## Step 0 — Install Dependencies

In [ ]:
# Run this cell once. Restart runtime after if prompted.
!pip install -q chromadb sentence-transformers pandas tqdm

---
## Step 1 — Mount Google Drive & Set Paths

In [ ]:
import os

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Path Configuration ──────────────────────────────────────────────────────
# Update DRIVE_PROJECT_ROOT to wherever you placed the Socratic_OT folder
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/Socratic_OT'   # <-- adjust if needed

CHUNKS_CSV       = os.path.join(DRIVE_PROJECT_ROOT, 'Data/text_chunks/text_chunks_full 2.csv')
IMAGE_META_JSON  = os.path.join(DRIVE_PROJECT_ROOT, 'Data/image_metadata/image_metadata.json')
CHROMA_PERSIST   = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db')   # where ChromaDB saves to disk

os.makedirs(CHROMA_PERSIST, exist_ok=True)

print('✅ Paths configured')
print(f'   Chunks CSV     : {CHUNKS_CSV}')
print(f'   Image metadata : {IMAGE_META_JSON}')
print(f'   ChromaDB dir   : {CHROMA_PERSIST}')

---
## Step 2 — Load & Inspect Text Chunks

In [ ]:
import pandas as pd

# ── Load the full chunk CSV ─────────────────────────────────────────────────
df = pd.read_csv(CHUNKS_CSV, dtype=str).fillna('')

print(f'Total chunks loaded : {len(df)}')
print(f'Columns             : {list(df.columns)}')
print()

# ── Chapter distribution ────────────────────────────────────────────────────
ch_dist = df.groupby('chapter').size().reset_index(name='chunks')
ch_dist['chapter'] = ch_dist['chapter'].astype(int)
ch_dist = ch_dist.sort_values('chapter')
print('Chapter distribution:')
print(ch_dist.to_string(index=False))
print()
print(f'Chapters covered : {sorted(df["chapter"].astype(int).unique())}')
print(f'Unique sections  : {df["section"].nunique()}')
print(f'Unique topics    : {df["topic"].nunique()}')

In [ ]:
# Inspect a sample chunk
sample = df.iloc[50]
print('=== Sample Chunk ===')
for col in df.columns:
    val = sample[col]
    if col == 'chunk_text':
        print(f'{col}:\n{val[:400]}...')
    else:
        print(f'{col}: {val}')

---
## Step 3 — Data Cleaning & Validation

Before embedding, filter out low-quality chunks (table-of-contents stubs, review question lists, etc.)

In [ ]:
import re

def is_low_quality(text: str) -> bool:
    """Return True for chunks that are mostly navigation/stub content."""
    text = text.strip()
    # Too short to be useful
    if len(text.split()) < 40:
        return True
    # Mostly keyword-list / review-question boilerplate
    boilerplate_patterns = [
        r'^(Key Terms|Chapter Review|Review Questions|Interactive Link)',
        r'(Key Terms\s*Chapter Review\s*Interactive)',
        r'(LEARNING OBJECTIVES\s*By the end of this section)',
    ]
    for pat in boilerplate_patterns:
        if re.search(pat, text[:200], re.IGNORECASE):
            # Allow if it has substantial body text after the header
            body = re.sub(r'.*(By the end of this section.*?:\s*)', '', text, flags=re.DOTALL)
            if len(body.split()) < 50:
                return True
    return False

# Apply filter
before = len(df)
df['_low_quality'] = df['chunk_text'].apply(is_low_quality)
df_clean = df[~df['_low_quality']].drop(columns=['_low_quality']).reset_index(drop=True)

print(f'Chunks before cleaning : {before}')
print(f'Chunks after cleaning  : {len(df_clean)}')
print(f'Removed (low quality)  : {before - len(df_clean)}')

---
## Step 4 — Load Embedding Model

Using `all-MiniLM-L6-v2` — free, local, no API key required, 384-dimensional embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# Use GPU if available (Colab T4 is free)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

print('Loading all-MiniLM-L6-v2 ...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f'✅ Embedder loaded  |  embedding dim = {embedder.get_sentence_embedding_dimension()}')

---
## Step 5 — Generate Embeddings

Embedding all ~997 chunks in batches. Takes ~1-2 minutes on Colab CPU, ~20 seconds on GPU.

In [ ]:
from tqdm.auto import tqdm
import numpy as np

BATCH_SIZE = 64

texts = df_clean['chunk_text'].tolist()

print(f'Embedding {len(texts)} chunks in batches of {BATCH_SIZE} ...')

all_embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding'):
    batch = texts[i : i + BATCH_SIZE]
    batch_emb = embedder.encode(
        batch,
        normalize_embeddings=True,   # cosine similarity via dot product
        show_progress_bar=False
    )
    all_embeddings.append(batch_emb)

embeddings = np.vstack(all_embeddings)
print(f'✅ Embeddings shape: {embeddings.shape}')   # (n_chunks, 384)

---
## Step 6 — Initialize ChromaDB & Ingest

Persisting to Google Drive so we never need to re-embed.

In [ ]:
import chromadb
from chromadb.config import Settings

# ── Persistent ChromaDB client ──────────────────────────────────────────────
client = chromadb.PersistentClient(path=CHROMA_PERSIST)

COLLECTION_NAME = 'socratic_ot_textbook'

# Delete existing collection if re-running (clean slate)
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f'Deleted existing collection: {COLLECTION_NAME}')

# Create new collection (cosine similarity)
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)

print(f'✅ ChromaDB collection created: {COLLECTION_NAME}')

In [ ]:
# ── Ingest in batches of 500 ────────────────────────────────────────────────
# ChromaDB recommends batches ≤ 5000; 500 is safe and fast

INGEST_BATCH = 500
total = len(df_clean)

print(f'Ingesting {total} chunks into ChromaDB ...')

for start in tqdm(range(0, total, INGEST_BATCH), desc='Ingesting'):
    end = min(start + INGEST_BATCH, total)
    batch_df  = df_clean.iloc[start:end]
    batch_emb = embeddings[start:end]

    collection.add(
        ids        = batch_df['chunk_id'].tolist(),
        embeddings = batch_emb.tolist(),
        documents  = batch_df['chunk_text'].tolist(),
        metadatas  = [
            {
                'source_id' : row['source_id'],
                'chapter'   : row['chapter'],
                'section'   : row['section'],
                'topic'     : row['topic'],
                'keywords'  : row['keywords'],
            }
            for _, row in batch_df.iterrows()
        ]
    )

print(f'✅ Ingestion complete')
print(f'   Chunks in collection: {collection.count()}')

---
## Step 7 — Build Retriever Function

This is the core retrieval function used by all downstream notebooks.

In [ ]:
from typing import Optional

def retrieve(
    query: str,
    top_k: int = 5,
    chapter_filter: Optional[str] = None,
    topic_filter: Optional[str] = None,
    merge_adjacent: bool = True
) -> list[dict]:
    """
    Retrieve the most relevant chunks from ChromaDB for a given query.

    Args:
        query           : The student's question or topic to look up
        top_k           : Number of results to retrieve (default 5)
        chapter_filter  : Optionally restrict to a specific chapter (e.g. '12')
        topic_filter    : Optionally restrict by topic label
        merge_adjacent  : If True, merge chunks from the same section to reduce fragmentation

    Returns:
        List of dicts with keys: chunk_id, score, section, chapter, topic, text
    """
    # Embed the query
    query_emb = embedder.encode(query, normalize_embeddings=True).tolist()

    # Build optional metadata filter
    where_clause = None
    if chapter_filter and topic_filter:
        where_clause = {'$and': [{'chapter': chapter_filter}, {'topic': topic_filter}]}
    elif chapter_filter:
        where_clause = {'chapter': chapter_filter}
    elif topic_filter:
        where_clause = {'topic': topic_filter}

    # Query ChromaDB
    kwargs = dict(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )
    if where_clause:
        kwargs['where'] = where_clause

    results = collection.query(**kwargs)

    # Format output
    hits = []
    for doc, meta, dist, cid in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
        results['ids'][0]
    ):
        hits.append({
            'chunk_id' : cid,
            'score'    : round(1 - dist, 4),   # cosine similarity (higher = better)
            'chapter'  : meta.get('chapter', ''),
            'section'  : meta.get('section', ''),
            'topic'    : meta.get('topic', ''),
            'keywords' : meta.get('keywords', ''),
            'text'     : doc
        })

    # Optional: merge chunks from the same section (reduce fragmentation)
    if merge_adjacent and hits:
        merged = {}
        for h in hits:
            key = h['section']
            if key not in merged:
                merged[key] = h.copy()
            else:
                # Keep highest score; append text if not duplicate
                if h['score'] > merged[key]['score']:
                    merged[key]['score'] = h['score']
                if h['text'] not in merged[key]['text']:
                    merged[key]['text'] += '\n\n' + h['text']
        hits = sorted(merged.values(), key=lambda x: x['score'], reverse=True)

    return hits[:3]   # Return top 2-3 final context units


print('✅ retrieve() function defined')

---
## Step 8 — Test Retrieval

Run sample queries spanning multiple chapters to confirm the vector store is working correctly.

In [ ]:
def show_results(query: str, **kwargs):
    print(f'\n{'='*70}')
    print(f'QUERY: "{query}"')
    print('='*70)
    results = retrieve(query, **kwargs)
    for i, r in enumerate(results, 1):
        print(f'\n--- Result {i} | Score: {r["score"]} | Chapter {r["chapter"]} | {r["section"]} ---')
        print(r['text'][:400] + ('...' if len(r['text']) > 400 else ''))

# Test 1: Nervous system (Chapter 12)
show_results('What is the function of the median nerve and what happens when it is compressed?')

# Test 2: Muscle anatomy (Chapter 10-11)
show_results('Describe the structure of skeletal muscle and the role of sarcomeres in contraction')

# Test 3: Clinical relevance
show_results('How does a stroke affect motor function and which brain areas are involved?')

# Test 4: Chapter-filtered query
show_results('Describe the brachial plexus and its branches', chapter_filter='13')

In [ ]:
# Test 5: Topics relevant to OT certification (NBCOT)
show_results('What is spasticity and how does it affect occupational performance?')

# Test 6: General anatomy (should retrieve from early chapters)
show_results('Explain the anatomical planes and directional terms used in clinical practice')

# Test 7: Verify cross-chapter breadth
show_results('What role does the endocrine system play in regulating metabolism and energy?')

---
## Step 9 — Load & Verify Image Metadata

In [ ]:
import json

with open(IMAGE_META_JSON, 'r') as f:
    image_metadata = json.load(f)

print(f'Image metadata records: {len(image_metadata)}')
print()

# Show first 3 records
for rec in image_metadata[:3]:
    print(json.dumps(rec, indent=2))
    print()

In [ ]:
# Build a fast lookup dict: topic → list of image records
from collections import defaultdict

image_by_topic = defaultdict(list)
image_by_structure = {}   # structure_name (lowercase) → record

for rec in image_metadata:
    topic = rec.get('topic', '').lower()
    structure = rec.get('structure_name', '').lower()
    image_by_topic[topic].append(rec)
    image_by_structure[structure] = rec
    # Also index by aliases
    for alias in rec.get('aliases', []):
        image_by_structure[alias.lower()] = rec

print('Topics with images:', list(image_by_topic.keys()))
print('Structures indexed:', list(image_by_structure.keys())[:10])


def find_image_for_topic(topic_or_structure: str) -> list:
    """
    Return image metadata records matching a topic or structure name.
    Used by the VLM module (Phase 4) to link text context to images.
    """
    q = topic_or_structure.lower()
    results = []
    # Exact structure match first
    if q in image_by_structure:
        results.append(image_by_structure[q])
    # Topic match
    for topic, recs in image_by_topic.items():
        if q in topic or topic in q:
            results.extend(recs)
    # Deduplicate
    seen = set()
    unique = []
    for r in results:
        key = r.get('image_id', r.get('file_name', str(r)))
        if key not in seen:
            seen.add(key)
            unique.append(r)
    return unique

# Test
print('\nTest find_image_for_topic("neuron"):')
matches = find_image_for_topic('neuron')
for m in matches:
    print(f'  {m.get("structure_name")} — {m.get("file_name")}')

print('\n✅ Image lookup ready')

---
## Step 10 — Save Retriever State for Downstream Notebooks

In [ ]:
import pickle

# Save image lookup dicts for use in later notebooks
state_path = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db/image_lookup.pkl')
with open(state_path, 'wb') as f:
    pickle.dump({'image_by_topic': dict(image_by_topic),
                 'image_by_structure': image_by_structure}, f)

print(f'✅ Image lookup saved to: {state_path}')
print()
print('ChromaDB is persisted automatically at:', CHROMA_PERSIST)
print()
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('PHASE 1 COMPLETE')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Chunks embedded & stored : {collection.count()}')
print(f'  Chapters covered         : 1–28 (full OpenStax A&P 2e)')
print(f'  Embedding model          : all-MiniLM-L6-v2 (384-dim, cosine)')
print(f'  Vector DB                : ChromaDB (persisted)')
print(f'  Image records indexed    : {len(image_metadata)}')
print()
print('→ Next: Run 02_tutoring_policy.ipynb')

---
## Step 11 — Quick Retrieval Quality Check

Score retrieval quality by checking that queries return chunks from expected chapters.

In [ ]:
# Ground-truth spot-checks: each query should return results from the expected chapter
quality_tests = [
    {'query': 'action potential and neuron membrane voltage',  'expected_chapter': '12'},
    {'query': 'rotator cuff muscles and shoulder joint',       'expected_chapter': '11'},
    {'query': 'cardiac output and stroke volume',              'expected_chapter': '19'},
    {'query': 'anatomy directional terms superior inferior',   'expected_chapter': '1'},
    {'query': 'lymphatic system and immune response',          'expected_chapter': '21'},
    {'query': 'female reproductive system and menstrual cycle','expected_chapter': '27'},
]

passed = 0
print('=== Retrieval Quality Check ===')
for test in quality_tests:
    results = retrieve(test['query'], top_k=5, merge_adjacent=False)
    top_chapters = [r['chapter'] for r in results]
    hit = test['expected_chapter'] in top_chapters
    status = '✅ PASS' if hit else '⚠️  MISS'
    if hit:
        passed += 1
    print(f'{status} | Ch{test["expected_chapter"]} | Top results from chapters: {top_chapters}')
    print(f'       Query: "{test["query"][:60]}"')

print(f'\nScore: {passed}/{len(quality_tests)} spot-checks passed')